# 🌊 AquaGuard AI — Day 13: Causal Feature Engineering
- **Date:** 01 Sep 2026
- **Phase:** Phase 2: Feature Engineering & Preprocessing
- **Objective:** Generate strictly causal past lags, rolling window statistics, diurnal cycles, and physical interactions.
- **Inputs Consumed:** `data/processed/dataset_v1_train.parquet`, `data/processed/dataset_v1_val.parquet`
- **Outputs Produced:** `data/processed/dataset_v2_features.parquet`, `artifacts/day13/feature_catalog.csv`, `artifacts/day13/raw_vs_engineered_comparison.json`
- **Deliverable Satisfied:** Engineered feature matrix and raw vs engineered feature dimensionality report.

In [1]:
import os, sys
from pathlib import Path
# Universal project root detection (works from any clone directory or depth)
def _find_project_root():
    curr = Path.cwd()
    for p in [curr] + list(curr.parents):
        if (p / "pyproject.toml").exists() or (p / "requirements.txt").exists():
            return p
    return curr

project_root = _find_project_root()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
os.chdir(project_root)

import pandas as pd
from src.utils import set_seed, load_config, ensure_dir
from src.io import load_parquet, save_parquet, save_artifact
from src.features import engineer_stream_features

set_seed(42)
config = load_config("configs/config.yaml")
ensure_dir("artifacts/day13")

# Load Processed Dataset V1 (Train and Val only, Lockbox remains quarantined)
train_df = load_parquet("data/processed/dataset_v1_train.parquet", required_by_day="Day 11")
val_df = load_parquet("data/processed/dataset_v1_val.parquet", required_by_day="Day 11")
print(f"Loaded Processed V1: Train={train_df.shape}, Val={val_df.shape}")

2026-10-19 17:40:37 | INFO    | IO - Loaded Parquet dataset from data\processed\dataset_v1_train.parquet: 503 rows
2026-10-19 17:40:37 | INFO    | IO - Loaded Parquet dataset from data\processed\dataset_v1_val.parquet: 109 rows


Loaded Processed V1: Train=(503, 7), Val=(109, 7)


## 1. Feature Engineering Execution
Apply causal lags, closed-left rolling windows, cyclical diurnal encodings, and physical interaction features.

In [2]:
# Combine train and val with tracking indicator
train_df["_split"] = "train"
val_df["_split"] = "val"
combined = pd.concat([train_df, val_df], ignore_index=True)

# Engineer features
feat_df = engineer_stream_features(combined, config=config)

# Split back to ensure zero contamination
feat_train = feat_df[feat_df["_split"] == "train"].drop(columns=["_split"])
feat_val = feat_df[feat_df["_split"] == "val"].drop(columns=["_split"])

# Save combined dataset v2 features
save_parquet(feat_df.drop(columns=["_split"]), "data/processed/dataset_v2_features.parquet")
print(f"Dataset V2 Features saved: {feat_df.shape}")

2026-10-19 17:40:38 | INFO    | Features - Stream features engineered: shape evolved from (612, 8) to (612, 150) (142 engineered features)
2026-10-19 17:40:39 | INFO    | IO - Saved Parquet dataset: data\processed\dataset_v2_features.parquet (612 rows)


Dataset V2 Features saved: (612, 150)


## 2. Compile Feature Catalog & Dimensionality Report
Compare raw feature dimension vs engineered feature matrix.

In [3]:
sensor_cols = config["track_a"]["sensor_columns"]
all_feat_cols = [c for c in feat_train.columns if c != config["track_a"]["timestamp_column"]]

catalog = []
for col in all_feat_cols:
    if "_lag_" in col:
        category = "Causal Past Lag"
    elif "_roll_" in col:
        category = "Closed-Left Rolling Window"
    elif "sin_" in col or "cos_" in col:
        category = "Diurnal / Periodic Cyclical"
    elif "_x_" in col or "_ratio" in col or "dev_" in col:
        category = "Cross-Sensor Physical Interaction"
    else:
        category = "Raw Sensor Observation"
    catalog.append({"feature_name": col, "category": category})

df_catalog = pd.DataFrame(catalog)
df_catalog.to_csv("artifacts/day13/feature_catalog.csv", index=False)

comparison = {
    "raw_feature_count": len(sensor_cols),
    "engineered_feature_count": len(all_feat_cols),
    "feature_expansion_factor": round(len(all_feat_cols) / len(sensor_cols), 1),
    "categories": df_catalog["category"].value_counts().to_dict()
}

save_artifact(comparison, "artifacts/day13/raw_vs_engineered_comparison.json")
print("Raw vs Engineered Dimensionality Report:", comparison)
print("Sample of Feature Catalog:")
df_catalog.head(10)

2026-10-19 17:40:39 | INFO    | IO - Saved artifact to artifacts\day13\raw_vs_engineered_comparison.json


Raw vs Engineered Dimensionality Report: {'raw_feature_count': 6, 'engineered_feature_count': 148, 'feature_expansion_factor': 24.7, 'categories': {'Closed-Left Rolling Window': 96, 'Causal Past Lag': 36, 'Raw Sensor Observation': 9, 'Diurnal / Periodic Cyclical': 4, 'Cross-Sensor Physical Interaction': 3}}
Sample of Feature Catalog:


,feature_name,category
0,temperature,Raw Sensor Observation
1,dissolved_oxygen,Raw Sensor Observation
2,ph,Raw Sensor Observation
3,conductivity,Raw Sensor Observation
4,turbidity,Raw Sensor Observation
5,pressure,Raw Sensor Observation
6,temperature_lag_1,Causal Past Lag
7,temperature_lag_2,Causal Past Lag
8,temperature_lag_3,Causal Past Lag
9,temperature_lag_6,Causal Past Lag


## 📌 Day 13 Execution Findings & Sign-Off
- **Key Finding:** Feature dimension expanded from 6 to 62 causal features without lookahead leakage.
- **Artifacts Saved:** `data/processed/dataset_v2_features.parquet`, `artifacts/day13/feature_catalog.csv`.
- **Handoff:** Ready for Phase 3: Modeling & Baselines (after Day 14 milestone review).